In [58]:
import os
import warnings

In [59]:
import numpy   as np
import pandas  as pd
import seaborn as sns

In [60]:
from scipy import stats
from scipy.stats import kruskal
from scipy.stats import anderson 

In [61]:
df_spec_data = pd.read_csv("../data/pre_processed/SDSS_DR16Q_spec.csv")

In [62]:
df_clean_spec = df_spec_data.copy()

# 1. Convertir ceros exactos a NaN de forma vectorizada
cols_zeros_to_nan = [
    'HALPHA_BR_FWHM', 'HALPHA_BR_EW', 'HBETA_BR_FWHM', 'HBETA_BR_EW', 
    'OIII5007_FWHM', 'OIII5007_EW', 'NII6585_FWHM', 'NII6585_EW', 
    'SII6718_FWHM', 'SII6718_EW', 'MGII_BR_EW', 'MGII_BR_FWHM', 
    'CIV_EW', 'CIV_FWHM', 'LOGMBH', 'LOGLBOL'
]

df_clean_spec[cols_zeros_to_nan] = df_clean_spec[cols_zeros_to_nan].replace(0.0, np.nan)

In [63]:
# 3. Filtrar Anchos a Media Altura (FWHM)
# Un FWHM > 40000 km/s es altamente sospechoso incluso para líneas muy anchas.
fwhm_cols = [col for col in df_clean_spec.columns if 'FWHM' in col]
df_clean_spec[fwhm_cols] = df_clean_spec[fwhm_cols].where(
    (df_clean_spec[fwhm_cols] > 0) & (df_clean_spec[fwhm_cols] <= 40000)
)

In [64]:
# 2. Filtrar Anchos Equivalentes (EW) irreales
ew_cols = [col for col in df_clean_spec.columns if 'EW' in col]
df_clean_spec[ew_cols] = df_clean_spec[ew_cols].where(
    (df_clean_spec[ew_cols] >= 0) & (df_clean_spec[ew_cols] <= 5000)
)

In [65]:
# 4. Limpiar variables físicas derivadas (Masa y Luminosidad)
df_clean_spec['LOGMBH']  = df_clean_spec['LOGMBH'].where(df_clean_spec['LOGMBH'] > 5.0)
df_clean_spec['LOGLBOL'] = df_clean_spec['LOGLBOL'].where(df_clean_spec['LOGLBOL'] > 40.0)

In [66]:
print(df_clean_spec.describe())

         FEII_OPT_EW  HALPHA_BR_FWHM  HALPHA_BR_EW  HBETA_BR_FWHM  \
count  732632.000000    24094.000000  2.404800e+04  159321.000000   
mean       24.693511     3675.053218  2.250982e+02    4201.758096   
std        97.959660     2603.594623  1.877401e+02    2642.189840   
min         0.000000     1392.632690  2.329537e-07    1397.867685   
25%         0.000000     2042.326346  1.127016e+02    2312.975049   
50%         0.000000     3049.272579  1.919639e+02    3608.916163   
75%        30.163054     4445.141619  2.920191e+02    5322.052253   
max      4946.005532    37020.933396  4.978221e+03   36082.119386   

        HBETA_BR_EW  OIII5007_FWHM   OIII5007_EW  NII6585_FWHM    NII6585_EW  \
count  1.530720e+05  142505.000000  1.397120e+05  22111.000000  2.187900e+04   
mean   9.380250e+01     581.359985  4.560171e+01    467.004407  2.513323e+01   
std    2.547882e+02     480.272281  1.813069e+02    264.044296  1.458602e+02   
min    3.770813e-13     162.623135  1.498722e-13    167.52

In [67]:
df_community_0 = pd.read_csv("../data/communities/by_color/df_community_0.csv")
df_community_1 = pd.read_csv("../data/communities/by_color/df_community_1.csv")
df_community_2 = pd.read_csv("../data/communities/by_color/df_community_2.csv")
df_community_3 = pd.read_csv("../data/communities/by_color/df_community_3.csv")
df_community_4 = pd.read_csv("../data/communities/by_color/df_community_4.csv")
df_community_5 = pd.read_csv("../data/communities/by_color/df_community_5.csv")
df_community_6 = pd.read_csv("../data/communities/by_color/df_community_6.csv")
df_community_7 = pd.read_csv("../data/communities/by_color/df_community_7.csv")

In [68]:
df_community_0 = pd.merge(df_community_0, df_spec_data, on="SDSS_NAME", how="left")
df_community_1 = pd.merge(df_community_1, df_spec_data, on="SDSS_NAME", how="left")
df_community_2 = pd.merge(df_community_2, df_spec_data, on="SDSS_NAME", how="left")
df_community_3 = pd.merge(df_community_3, df_spec_data, on="SDSS_NAME", how="left")
df_community_4 = pd.merge(df_community_4, df_spec_data, on="SDSS_NAME", how="left")
df_community_5 = pd.merge(df_community_5, df_spec_data, on="SDSS_NAME", how="left")
df_community_6 = pd.merge(df_community_6, df_spec_data, on="SDSS_NAME", how="left")
df_community_7 = pd.merge(df_community_7, df_spec_data, on="SDSS_NAME", how="left")


In [69]:
dfs = [
    df_community_0, df_community_1, df_community_2, df_community_3,
    df_community_4, df_community_5, df_community_6, df_community_7,
]

df_all = pd.concat(dfs)

In [80]:
len(df_all["Z"])

5771

In [82]:
features = [
    'FEII_OPT_EW', 'HALPHA_BR_FWHM', 'HALPHA_BR_EW', 'HBETA_BR_FWHM', 'HBETA_BR_EW',
    'OIII5007_FWHM', 'OIII5007_EW', 'NII6585_FWHM', 'NII6585_EW',
    'SII6718_FWHM', 'SII6718_EW', 'MGII_BR_EW', 'MGII_BR_FWHM', 'CIV_EW',
    'CIV_FWHM', 'LOGMBH', 'LOGLBOL', 'LOGLEDD_RATIO'
]

for feature in features:
    communities_list = sorted(df_all['community'].unique())
    
    communities_data = {}
    N_total = 0 # Contador para el total de datos válidos reales
    
    for c in communities_list:
        # Extraemos solo los valores válidos (sin NaNs) para esta comunidad y este feature
        datos_validos = df_all[df_all['community'] == c][feature].dropna().values
        communities_data[c] = datos_validos
        N_total += len(datos_validos) # Sumamos al gran total N

    # Ejecutamos Kruskal-Wallis desempaquetando los arrays limpios
    stat_kw, p_kw = stats.kruskal(*communities_data.values())
    
    # Ahora sí, dividimos por (N - 1)
    epsilon_sq = stat_kw / (N_total - 1) if N_total > 1 else np.nan
    
    print(f"Kruskal-Wallis global for {feature}: H={stat_kw:.2f}, p-valor={p_kw:.2e}, e2={epsilon_sq:.4f}")

Kruskal-Wallis global for FEII_OPT_EW: H=1874.89, p-valor=0.00e+00, e2=0.3249
Kruskal-Wallis global for HALPHA_BR_FWHM: H=2306.08, p-valor=0.00e+00, e2=0.3997
Kruskal-Wallis global for HALPHA_BR_EW: H=2363.35, p-valor=0.00e+00, e2=0.4096
Kruskal-Wallis global for HBETA_BR_FWHM: H=2211.57, p-valor=0.00e+00, e2=0.3833
Kruskal-Wallis global for HBETA_BR_EW: H=2450.74, p-valor=0.00e+00, e2=0.4247
Kruskal-Wallis global for OIII5007_FWHM: H=2211.83, p-valor=0.00e+00, e2=0.3833
Kruskal-Wallis global for OIII5007_EW: H=2267.52, p-valor=0.00e+00, e2=0.3930
Kruskal-Wallis global for NII6585_FWHM: H=2032.75, p-valor=0.00e+00, e2=0.3523
Kruskal-Wallis global for NII6585_EW: H=2057.16, p-valor=0.00e+00, e2=0.3565
Kruskal-Wallis global for SII6718_FWHM: H=2106.66, p-valor=0.00e+00, e2=0.3651
Kruskal-Wallis global for SII6718_EW: H=2102.16, p-valor=0.00e+00, e2=0.3643
Kruskal-Wallis global for MGII_BR_EW: H=949.71, p-valor=8.82e-201, e2=0.1646
Kruskal-Wallis global for MGII_BR_FWHM: H=1016.06, p-valo

In [74]:
matrices_ad = {}

# Extraemos las comunidades tal cual vienen, sin renombrar nada
communities_list = sorted(df_all['community'].unique())
n_com = len(communities_list)

for feature in features:
    # SOLUCIÓN: Creamos la matriz llena de ceros con np.zeros
    matriz = pd.DataFrame(
        np.zeros((n_com, n_com)), 
        index=communities_list, 
        columns=communities_list
    )
    
    # Extraemos los datos a un diccionario
    communities_data = {c: df_all[df_all['community'] == c][feature].dropna().values for c in communities_list}
    
    for c1, c2 in combinations(communities_list, 2):
        g1 = communities_data[c1]
        g2 = communities_data[c2]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            # anderson_ksamp devuelve: statistic, critical_values, pvalue
            res = stats.anderson_ksamp([g1, g2])
            stat_ad = res.statistic
            
        # Llenamos la matriz de forma simétrica usando .loc
        matriz.loc[c1, c2] = round(stat_ad, 2)
        matriz.loc[c2, c1] = round(stat_ad, 2)
        
    # Guardamos la matriz terminada
    matrices_ad[feature] = matriz

# Revisamos el resultado de uno de tus parámetros más fuertes
print("Anderson-Darling Matrix for LOGMBH:")
print(matrices_ad['LOGMBH'])

Anderson-Darling Matrix for LOGMBH:
        0       1       2       3       4       5       6       7
0    0.00   34.93  530.70  444.06  179.01   34.85   70.37   10.74
1   34.93    0.00  520.60  496.72  102.44   10.09   45.12   35.82
2  530.70  520.60    0.00  212.48  211.22  292.08  187.21  541.65
3  444.06  496.72  212.48    0.00  382.18  355.97  266.43  433.34
4  179.01  102.44  211.22  382.18    0.00   46.63   34.14  204.58
5   34.85   10.09  292.08  355.97   46.63    0.00   11.19   50.96
6   70.37   45.12  187.21  266.43   34.14   11.19    0.00   89.14
7   10.74   35.82  541.65  433.34  204.58   50.96   89.14    0.00


In [75]:
# Diccionarios para guardar las nuevas matrices
matrices_cliff = {}
matrices_dif_mediana = {}
matrices_mwu_p = {} # Por si tu guía te pide reportar los p-valores de Mann-Whitney

communities_list = sorted(df_all['community'].unique())
n_com = len(communities_list)

for feature in features:
    # Inicializamos las matrices con ceros
    matriz_cliff = pd.DataFrame(np.zeros((n_com, n_com)), index=communities_list, columns=communities_list)
    matriz_mediana = pd.DataFrame(np.zeros((n_com, n_com)), index=communities_list, columns=communities_list)
    matriz_p = pd.DataFrame(np.zeros((n_com, n_com)), index=communities_list, columns=communities_list)
    
    # Extraemos los datos omitiendo NaNs
    communities_data = {c: df_all[df_all['community'] == c][feature].dropna().values for c in communities_list}
    
    for c1, c2 in combinations(communities_list, 2):
        g1 = communities_data[c1]
        g2 = communities_data[c2]
        n1, n2 = len(g1), len(g2)
        
        # 1. Test de Mann-Whitney U
        # Calculamos U1 (el estadístico para g1)
        res_mwu = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        u1 = res_mwu.statistic
        p_val = res_mwu.pvalue
        
        # 2. Cálculo rápido de Delta de Cliff usando U1
        # Fórmula: d = (2 * U1 / (n1 * n2)) - 1
        d_cliff = (2 * u1 / (n1 * n2)) - 1
        
        # 3. Diferencia Absoluta de Medianas
        dif_mediana = np.abs(np.median(g1) - np.median(g2))
        
        # Llenamos las matrices (Delta de Cliff es direccional, por eso usamos valor absoluto para la matriz simétrica)
        # Si prefieres ver la dirección (qué comunidad es mayor), puedes quitar el np.abs()
        matriz_cliff.loc[c1, c2] = round(np.abs(d_cliff), 3)
        matriz_cliff.loc[c2, c1] = round(np.abs(d_cliff), 3)
        
        matriz_mediana.loc[c1, c2] = round(dif_mediana, 3)
        matriz_mediana.loc[c2, c1] = round(dif_mediana, 3)
        
        matriz_p.loc[c1, c2] = p_val
        matriz_p.loc[c2, c1] = p_val
        
    # Guardamos en los diccionarios
    matrices_cliff[feature] = matriz_cliff
    matrices_dif_mediana[feature] = matriz_mediana
    matrices_mwu_p[feature] = matriz_p

# Muestra los resultados de tu parámetro estrella
print("Matriz Delta de Cliff (|d|) para LOGMBH:")
print(matrices_cliff['LOGMBH'])

Matriz Delta de Cliff (|d|) para LOGMBH:
       0      1      2      3      4      5      6      7
0  0.000  0.206  0.756  0.925  0.507  0.244  0.321  0.012
1  0.206  0.000  0.701  0.921  0.365  0.069  0.178  0.236
2  0.756  0.701  0.000  0.562  0.471  0.589  0.421  0.840
3  0.925  0.921  0.562  0.000  0.840  0.863  0.729  0.970
4  0.507  0.365  0.471  0.840  0.000  0.246  0.095  0.593
5  0.244  0.069  0.589  0.863  0.246  0.000  0.112  0.271
6  0.321  0.178  0.421  0.729  0.095  0.112  0.000  0.352
7  0.012  0.236  0.840  0.970  0.593  0.271  0.352  0.000


In [83]:
print("\nMatriz Diferencia de Medianas para LOGMBH:")
print(matrices_dif_mediana['LOGMBH'])


Matriz Diferencia de Medianas para LOGMBH:
       0      1      2      3      4      5      6      7
0  0.000  0.167  0.831  1.374  0.407  0.211  0.305  0.044
1  0.167  0.000  0.664  1.207  0.240  0.044  0.139  0.123
2  0.831  0.664  0.000  0.543  0.424  0.620  0.525  0.787
3  1.374  1.207  0.543  0.000  0.967  1.163  1.069  1.330
4  0.407  0.240  0.424  0.967  0.000  0.196  0.102  0.363
5  0.211  0.044  0.620  1.163  0.196  0.000  0.094  0.167
6  0.305  0.139  0.525  1.069  0.102  0.094  0.000  0.262
7  0.044  0.123  0.787  1.330  0.363  0.167  0.262  0.000
